# Dựng Knowledge Graph — nhánh PDF/web → KB

Notebook này là nhánh dưới của sơ đồ trên bảng: từ **trang web Nhà thuốc Long Châu** dựng **Knowledge Base dưới dạng
Knowledge Graph** trên Neo4j, để Knowledge Agent truy vấn (GraphRAG).

```
kb/raw_html/*.html → (1) tách mục → (2) chia chunk → (3) LLM trích xuất + kiểm chứng
                   → (4) chuẩn hóa / gộp thực thể → (5) cầu nối Feature ↔ KG → (6) nạp Neo4j → (7) kiểm tra
```

**Trước khi chạy:** đã lưu trang Long Châu vào `kb/raw_html/` (`python -m kb.open_urls` rồi `python -m kb.import_html`),
Ollama và Neo4j đang chạy. Bước 3 mất khoảng 30–40 phút ở lần đầu (20 trang); kết quả LLM được cache trong `kb/cache/` nên chạy lại
(hoặc chạy tiếp sau khi dừng giữa chừng) rất nhanh.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path(os.environ.get("PROJECT_DIR", Path.cwd())).resolve()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent          # tìm thư mục src/Project
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import json, random, time
from collections import Counter

import pandas as pd
from IPython.display import display

import config
from agents.llm import LLM, Usage
from kb import load_neo4j
from kb.extract import extract_chunk, section_tasks
from kb.feature_bridge import add_feature_bridge
from kb.normalize import build_graph, load_merge_decisions
from kb.parse import chunk_page, parse_page
from kb.seed import load_seeds
import nbstyle

nbstyle.install()           # bảng có màu cố định, đọc rõ ở cả giao diện sáng lẫn tối
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
llm = LLM()
print("Nguồn HTML:", config.RAW_HTML_DIR, "| Nhãn KG trong Neo4j:", config.KG_LABEL, "| LLM:", llm.model)

Nguồn HTML: /Users/macos/Docs/Kì 1 năm 4/PTHTTM/src/Project/kb/raw_html | Nhãn KG trong Neo4j: DKG | LLM: qwen3.5:9b


## Bước 1 — Tách mỗi trang thành các mục

Mỗi thẻ tiêu đề (h2/h3/h4) mở một mục; nội dung `p`/`li`/`td` phía sau thuộc mục đó. Menu, footer và các mục
"Bài viết liên quan", "Câu hỏi thường gặp"… bị loại. Thực thể chính của trang bệnh/thuốc lấy từ cột `topic` của seed.

In [2]:
seeds = [s for s in load_seeds() if (config.RAW_HTML_DIR / f"{s.slug}.html").exists()]
pages = []
for seed in seeds:
    html = (config.RAW_HTML_DIR / f"{seed.slug}.html").read_text(encoding="utf-8", errors="ignore")
    pages.append(parse_page(html, seed))
with open(config.PROCESSED_DIR / "pages.jsonl", "w", encoding="utf-8") as f:
    for page in pages:
        f.write(json.dumps(page, ensure_ascii=False) + "\n")

overview = pd.DataFrame([{"loại": p["type"], "tiêu đề": p["title"][:70], "thực thể chính": ", ".join(p["main_entities"]),
                          "số mục": len(p["sections"]), "số ký tự": sum(len(s["text"]) for s in p["sections"])} for p in pages])
print(f"{len(pages)}/{len(load_seeds())} trang đã có trong {config.RAW_HTML_DIR}")
display(overview.groupby("loại").agg(trang=("tiêu đề", "count"), mục_TB=("số mục", "mean"), ký_tự_TB=("số ký tự", "mean")).round(1))
overview.head(10)

20/20 trang đã có trong /Users/macos/Docs/Kì 1 năm 4/PTHTTM/src/Project/kb/raw_html


,trang,mục_TB,ký_tự_TB
loại,,,
bai-viet,12,10.8,7592.0
benh,5,16.8,14 405.8
thuoc,3,16.3,15 463.0


loại,tiêu đề,thực thể chính,số mục,số ký tự
benh,"Đái tháo đường: Triệu chứng, nguyên nhân, chẩn đoán và điều trị",Đái tháo đường,18,17 624
benh,"Tiểu đường tuýp 1: Nguyên nhân, triệu chứng, chẩn đoán và phương pháp",Đái tháo đường tuýp 1,17,14 224
benh,"Tiểu đường tuýp 2 là gì? Nguyên nhân, triệu chứng và cách điều trị",Đái tháo đường tuýp 2,18,12 893
benh,Hạ đường huyết là gì? Nguyên nhân và cách phòng ngừa,Hạ đường huyết,15,8282
benh,"Bệnh tim mạch: Triệu chứng, nguyên nhân và cách điều trị",Bệnh tim mạch,16,19 006
bai-viet,11 triệu chứng tiểu đường giai đoạn đầu phổ biến,,17,8951
bai-viet,Chỉ số đường huyết tiền tiểu đường là bao nhiêu?,,7,7707
bai-viet,Chỉ số HbA1c bình thường là bao nhiêu? Khi nào nên làm xét nghiệm HbA1,,41,15 052
bai-viet,Hướng dẫn cách đọc chỉ số đường huyết chính xác và đơn giản nhất,,12,7937
bai-viet,6 biến chứng bệnh tiểu đường thường gặp và cách phòng ngừa,,10,6808


In [3]:
# Các trang tách được quá ít mục cần kiểm tra lại (có thể lưu nhầm trang hoặc cấu trúc HTML khác)
overview[overview["số mục"] < 2]

loại,tiêu đề,thực thể chính,số mục,số ký tự


In [4]:
# Xem thử các mục của một trang bệnh
sample = next((p for p in pages if p["type"] == "benh"), pages[0])
pd.DataFrame([{"tiêu đề mục": s["heading"], "đầu nội dung": s["text"][:90]} for s in sample["sections"]])

tiêu đề mục,đầu nội dung
Giới thiệu,Bệnh tiểu đường (Đái tháo đường) là căn bệnh mãn tính ngày càng phổ biến và đang trẻ hoá d
Tìm hiểu chung đái tháo đường (tiểu đường),Tiểu đường hay đái tháo đường là một bệnh mạn tính gây ra do thiếu sản xuất insulin của tu
Một số loại đái tháo đường thường gặp,Tiểu đường (Đái tháo đường) thường được chia thành 3 loại thường gặp bao gồm đái tháo đườn
Những dấu hiệu và triệu chứng của bệnh tiểu đường,Triệu chứng của đái tháo đường tuýp 1 Triệu chứng lâm sàng thường được gọi là hội chứng 4
Biến chứng có thể gặp khi mắc bệnh tiểu đường,"Các biến chứng tiểu đường cấp tính thường do chẩn đoán muộn, điều trị không thích hợp hoặc"
Khi nào cần gặp bác sĩ?,"Nếu có bất kỳ triệu chứng nào nêu trên xảy ra, bạn nên liên hệ ngay với bác sĩ để được kiể"
Nguyên nhân đái tháo đường (tiểu đường),Nguyên nhân của đái tháo đường tuýp 1 Trong số các trường hợp bị đái tháo đường tuýp 1 thì
Những ai có nguy cơ mắc bệnh tiểu đường,• Tuổi ≥ 45; • Thừa cân hoặc béo phì ; • Lối sống ít vận động; • Tiền sử gia đình có người
Một số yếu tố làm tăng nguy cơ mắc bệnh tiểu đường,"• Hút thuốc lá; • Béo phì, ít vận động; • Phụ nữ mang thai,… Tìm hiểu thêm: Béo phì và tiể"
Phương pháp xét nghiệm và chẩn đoán bệnh tiểu đường,Bệnh tiểu đường được gợi ý bởi các triệu chứng và dấu hiệu điển hình và được chứng minh bằ


## Bước 2 — Chia chunk và xác định tác vụ trích xuất

Mỗi mục là một chunk (cắt theo đoạn nếu dài hơn ~320 từ). Với mỗi chunk:
- **guided**: tên mục khớp luật (vd. "Triệu chứng" → `HAS_SYMPTOM`, "Tác dụng phụ" → `HAS_SIDE_EFFECT`), LLM chỉ cần liệt kê thực thể;
- **generic**: mục/bài viết không khớp luật, LLM trích bộ ba theo ontology;
- **threshold**: đoạn có con số kèm đơn vị xét nghiệm, LLM trích ngưỡng chẩn đoán.
Mục "Liều dùng" của trang thuốc **cố ý bị bỏ qua** để hệ thống không bao giờ có dữ liệu liều thuốc.

In [5]:
chunks = [c for page in pages for c in chunk_page(page)]
task_counts = Counter(t["kind"] + (":" + t["relation"] if t.get("relation") else "") for c in chunks for t in section_tasks(c))
print(f"{len(chunks)} chunk, {sum(task_counts.values())} tác vụ LLM")
pd.Series(task_counts).sort_values(ascending=False).to_frame("số tác vụ")

312 chunk, 325 tác vụ LLM


,số tác vụ
generic,211
threshold,24
guided:TREATED_BY,18
guided:HAS_RISK_FACTOR,17
guided:HAS_SYMPTOM,11
guided:HAS_COMPLICATION,11
guided:HAS_SIDE_EFFECT,11
guided:PREVENTED_BY,9
guided:DIAGNOSED_BY,8
guided:CONTRAINDICATED_IN,3


## Bước 3 — Trích xuất bằng LLM và kiểm chứng bằng code

LLM nộp kết quả qua một "tool" có schema Pydantic. Sau đó code kiểm tra từng bản ghi:
1. quan hệ và loại thực thể đúng ontology (domain/range);
2. câu trích (`evidence`) phải có trong văn bản gốc (khớp ≥ 90%); nếu model ghép câu nhưng tên thực thể có nguyên văn,
   câu gốc chứa tên đó được dùng làm bằng chứng;
3. tên thực thể phải xuất hiện trong văn bản (chống tên bịa);
4. con số ngưỡng phải có trong câu trích.

Chạy thử trên vài trang trước để xem chất lượng, rồi mới chạy toàn bộ.

In [6]:
PILOT_PAGES = 5
pilot_urls = {p["url"] for p in pages[:PILOT_PAGES]}
usage, accepted, rejected, start = Usage(), [], [], time.time()
for chunk in (c for c in chunks if c["url"] in pilot_urls):
    a, r = extract_chunk(llm, chunk, usage)
    accepted += a; rejected += r
print(f"Pilot {PILOT_PAGES} trang: {len(accepted)} chấp nhận, {len(rejected)} loại, {time.time() - start:.0f}s, {usage.as_dict()}")
pd.DataFrame([{k: r[k] for k in ("head", "relation", "tail", "evidence")} for r in accepted]).head(15)

Pilot 5 trang: 583 chấp nhận, 14 loại, 0s, {'calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0, 'seconds': 0.0}


head,relation,tail,evidence
Đái tháo đường,HAS_RISK_FACTOR,Thiếu sản xuất insulin của tuỵ,Tiểu đường hay đái tháo đường là một bệnh mạn tính gây ra do thiếu sản xuất insulin của tuỵ
Đái tháo đường,HAS_RISK_FACTOR,Tác dụng insulin không hiệu quả,hoặc tác dụng insulin không hiệu quả do nguyên nhân mắc phải hoặc do di truyền
Đái tháo đường,HAS_RISK_FACTOR,Di truyền,do nguyên nhân mắc phải hoặc do di truyền
Đái tháo đường tuýp 1,HAS_RISK_FACTOR,Tế bào beta của tuyến tụy bị phá hủy,Nguyên nhân là do tế bào beta của tuyến tụy bị phá hủy gây giảm tiết insulin hoặc không tiết ra insulin
Đái tháo đường tuýp 1,HAS_RISK_FACTOR,Người trẻ,thường xảy ra ở người trẻ
Đái tháo đường tuýp 2,HAS_RISK_FACTOR,Người lớn tuổi,dạng bệnh tiểu đường thường gặp ở người lớn tuổi
Đái tháo đường tuýp 2,HAS_RISK_FACTOR,Trên 40 tuổi,bệnh thường gặp ở những người trên 40 tuổi
Đái tháo đường tuýp 2,HAS_RISK_FACTOR,Đề kháng insulin,do suy giảm chức năng của tế bào beta tuyến tụy tiến triển trên nền tảng đề kháng insulin
Đái tháo đường thai kỳ,HAS_RISK_FACTOR,"Hormone estrogen, progesterone","Khi mang thai, các thụ thể insulin ở trên tế bào đích sẽ bị tác động bởi các hormone estrogen, progesterone"
Đái tháo đường thứ phát,HAS_RISK_FACTOR,Gen bị khiếm khuyết,còn có một số loại đái tháo đường thứ phát do các nguyên nhân như: Gen bị khiếm khuyết


In [7]:
pd.DataFrame([{k: r.get(k) for k in ("head", "relation", "tail", "reason")} for r in rejected]).head(15)

head,relation,tail,reason
Đái tháo đường thai kỳ,HAS_RISK_FACTOR,Mang thai,sai miền: Disease -HAS_RISK_FACTOR-> Lifestyle
Đái tháo đường thai kỳ,HAS_COMPLICATION,Biến chứng nguy hiểm,tên quá chung chung
Hạ glucose máu,DIAGNOSED_BY,Đường huyết,tên thực thể không xuất hiện trong văn bản
Hạ glucose máu,DIAGNOSED_BY,Đường huyết,tên thực thể không xuất hiện trong văn bản
Glucose máu trước ăn,DIAGNOSED_BY,Glucose máu trước ăn,tên rỗng hoặc tự trỏ
Glucose máu trước ăn,DIAGNOSED_BY,Glucose máu trước ăn,tên rỗng hoặc tự trỏ
Glucose máu trước ăn,DIAGNOSED_BY,Glucose máu trước ăn,tên rỗng hoặc tự trỏ
Đỉnh glucose máu sau ăn,DIAGNOSED_BY,Đỉnh glucose máu sau ăn,tên rỗng hoặc tự trỏ
Đỉnh glucose máu sau ăn,DIAGNOSED_BY,Đỉnh glucose máu sau ăn,tên rỗng hoặc tự trỏ
HbA1C,DIAGNOSED_BY,HbA1C,tên rỗng hoặc tự trỏ


In [8]:
# Chạy toàn bộ (kết quả LLM đã cache sẽ không gọi lại). In tiến độ mỗi 20 chunk.
usage, accepted, rejected, start = Usage(), [], [], time.time()
for i, chunk in enumerate(chunks, start=1):
    a, r = extract_chunk(llm, chunk, usage)
    accepted += a; rejected += r
    if i % 20 == 0 or i == len(chunks):
        print(f"  {i}/{len(chunks)} chunk | {len(accepted)} chấp nhận | {len(rejected)} loại | {time.time() - start:.0f}s")
for name, rows in (("records.jsonl", accepted), ("rejected.jsonl", rejected)):
    with open(config.PROCESSED_DIR / name, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("LLM:", usage.as_dict())

  20/312 chunk | 114 chấp nhận | 10 loại | 0s


  40/312 chunk | 180 chấp nhận | 13 loại | 0s
  60/312 chunk | 307 chấp nhận | 13 loại | 0s
  80/312 chunk | 428 chấp nhận | 14 loại | 0s
  100/312 chunk | 576 chấp nhận | 14 loại | 0s
  120/312 chunk | 650 chấp nhận | 32 loại | 0s
  140/312 chunk | 710 chấp nhận | 43 loại | 0s
  160/312 chunk | 722 chấp nhận | 91 loại | 0s


  180/312 chunk | 781 chấp nhận | 112 loại | 0s


  200/312 chunk | 884 chấp nhận | 123 loại | 0s


  220/312 chunk | 937 chấp nhận | 149 loại | 0s
  240/312 chunk | 986 chấp nhận | 170 loại | 0s
  260/312 chunk | 1064 chấp nhận | 204 loại | 0s
  280/312 chunk | 1108 chấp nhận | 221 loại | 0s
  300/312 chunk | 1189 chấp nhận | 242 loại | 0s
  312/312 chunk | 1202 chấp nhận | 265 loại | 0s
LLM: {'calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0, 'seconds': 0.0}


In [9]:
# Lý do bị loại và tỉ lệ chấp nhận theo loại tác vụ
reasons = Counter(r["reason"].split(":")[0] for r in rejected)
by_task = pd.DataFrame({"chấp nhận": Counter(r["task"] for r in accepted), "loại": Counter(r["task"] for r in rejected)}).fillna(0).astype(int)
by_task["tỉ lệ chấp nhận"] = (by_task["chấp nhận"] / by_task.sum(axis=1)).round(3)
display(by_task)
pd.Series(reasons).sort_values(ascending=False).to_frame("số bản ghi bị loại")

,chấp nhận,loại,tỉ lệ chấp nhận
generic,548,251,0.686
guided,604,0,1.000
threshold,50,14,0.781


,số bản ghi bị loại
sai miền,181
tên thực thể không xuất hiện trong văn bản,49
tên quá chung chung,15
tên rỗng hoặc tự trỏ,6
câu trích không có trong văn bản,6
con số ngưỡng không có trong câu trích,4
tên là con số (ngưỡng phải nằm trong thuộc tính của DIAGNOSED_BY),2
đơn vị % chỉ dùng cho HbA1c,2


## Bước 4 — Chuẩn hóa và gộp thực thể

- Khóa không dấu + từ điển đồng nghĩa: *tiểu đường* = *đái tháo đường* = *ĐTĐ*; *type/tip/tuýp 2*; *cao huyết áp* = *tăng huyết áp*…
- Khóa vừa là RiskFactor vừa là Disease (vd. *Tăng huyết áp*) được thống nhất thành Disease.
- Hai thực thể cùng loại có cosine bge-m3 ≥ 0,95 thì gộp; 0,92–0,95 thì theo quyết định duyệt tay trong
  `kb/merge_decisions.csv` (cột `merge`: 1 gộp, 0 giữ riêng). Cặp chưa có quyết định ghi ra `merge_candidates.csv`.
  Không bao giờ gộp tên khác nhau về con số hoặc từ trái nghĩa (tuýp 1/tuýp 2, tăng/hạ…).

In [10]:
decisions = load_merge_decisions(config.KB_DIR / "merge_decisions.csv")   # duyệt tay các cặp cosine 0,92–0,95
graph = add_feature_bridge(build_graph(accepted, llm, decisions=decisions))
pd.DataFrame(graph["merge_candidates"]).to_csv(config.PROCESSED_DIR / "merge_candidates.csv", index=False)
json.dump(graph, open(config.PROCESSED_DIR / "graph.json", "w", encoding="utf-8"), ensure_ascii=False)

entities = pd.DataFrame(graph["entities"])
print(f"{len(entities)} thực thể, {len(graph['relations'])} quan hệ | đã duyệt {len(decisions)} cặp "
      f"({sum(decisions.values())} gộp), còn {len(graph['merge_candidates'])} cặp chưa duyệt")
display(entities.groupby("type").size().sort_values(ascending=False).to_frame("số thực thể"))
merged = entities[entities["aliases"].map(len) > 0].sort_values("mention_count", ascending=False)
merged[["type", "name", "aliases", "mention_count"]].head(15)

816 thực thể, 1052 quan hệ | đã duyệt 54 cặp (48 gộp), còn 0 cặp chưa duyệt


,số thực thể
type,
Disease,188
Lifestyle,172
Symptom,147
RiskFactor,112
SideEffect,91
Drug,46
Test,36
DrugClass,16
Feature,8


,type,name,aliases,mention_count
0,Disease,Đái tháo đường,"Bệnh tiểu đường, Bệnh đái tháo đường, Tiểu đường",370
7,Disease,Đái tháo đường tuýp 2,"Bệnh tiểu đường tuýp 2, Bệnh tiểu đường type 2, Bệnh đái tháo đường type 2, Tiểu đường tuýp 2, Tiểu đường type 2, Đái tháo đường không phụ thuộc insulin, Đái tháo đường type 2, Đái tháo đường típ 2, Đái tháo đường týp 2",195
333,Disease,Bệnh tim mạch,Tim mạch,170
29,Disease,Hạ đường huyết,Hạ glucose máu,101
4,Disease,Đái tháo đường tuýp 1,"Tiểu đường tuýp 1, Tiểu đường type 1, Đái tháo đường phụ thuộc insulin, Đái tháo đường type 1, Đái tháo đường týp 1",60
46,Disease,Cao huyết áp,"Huyết áp cao, Tăng huyết áp",38
71,Test,HbA1c,"HbA1C, Xét nghiệm HbA1c",32
707,Drug,Gliclazide,Gliclazid,31
115,Lifestyle,Tập thể dục thường xuyên,"Duy trì vận động thể chất đều đặn, Hoạt động thể chất đều đặn, Luyện tập thể dục thể thao thường xuyên, Tập luyện thể thao đều đặn, Tập thể dục đều đặn, Vận động thường xuyên",12
66,Test,Đường huyết lúc đói,"Xét nghiệm đường huyết lúc đói, Đường huyết đói",12


## Bước 5–6 — Cầu nối Feature và nạp vào Neo4j

8 cột của dataset trở thành nút `Feature`, nối `MAPS_TO` tới xét nghiệm / yếu tố nguy cơ tương ứng. Đây là chỗ hai nhánh
trên sơ đồ gặp nhau: SHAP nói *HbA1c đẩy nguy cơ lên* → `Feature(HbA1c_level)` → `Test(HbA1c)` → ngưỡng chẩn đoán kèm câu trích.

Mọi node mang nhãn `DKG`; nạp lại chỉ xóa các node này, không đụng dữ liệu khác trong Neo4j.

In [11]:
with load_neo4j.get_driver() as driver:
    removed = load_neo4j.reset(driver)
    load_neo4j.ensure_schema(driver)
    start = time.time()
    load_neo4j.load_graph(driver, graph, chunks, pages, llm)
    node_stats, rel_stats = load_neo4j.stats(driver)
print(f"Đã xóa {removed} node cũ, nạp lại trong {time.time() - start:.0f}s")
display(node_stats)
rel_stats

Đã xóa 1186 node cũ, nạp lại trong 23s


type,count
Chunk,312
Disease,188
Lifestyle,172
Symptom,147
RiskFactor,112
SideEffect,91
Drug,46
Test,36
Source,20
DrugClass,16


relation,count
MENTIONS,1387
FROM,312
HAS_RISK_FACTOR,218
HAS_SYMPTOM,170
TREATED_BY,166
PREVENTED_BY,139
HAS_COMPLICATION,106
HAS_SIDE_EFFECT,95
DIAGNOSED_BY,79
CONTRAINDICATED_IN,43


In [12]:
graph_rows = []
for rel in graph["relations"]:
    graph_rows.append({"relation": rel["relation"], "head": rel["head"].split(":")[0], "tail": rel["tail"].split(":")[0]})
pd.crosstab(pd.DataFrame(graph_rows)["relation"], pd.DataFrame(graph_rows)["tail"])

tail,Disease,Drug,DrugClass,Lifestyle,RiskFactor,SideEffect,Symptom,Test
relation,,,,,,,,
BELONGS_TO,0,0,4,0,0,0,0,0
CONTRAINDICATED_IN,43,0,0,0,0,0,0,0
DIAGNOSED_BY,0,0,0,0,0,0,0,79
HAS_COMPLICATION,106,0,0,0,0,0,0,0
HAS_RISK_FACTOR,76,0,0,0,142,0,0,0
HAS_SIDE_EFFECT,0,0,0,0,0,95,0,0
HAS_SYMPTOM,0,0,0,0,0,0,170,0
MAPS_TO,8,0,0,0,14,0,0,10
PREVENTED_BY,0,0,0,139,0,0,0,0


## Bước 7 — Kiểm tra bằng các công cụ mà agent sẽ dùng

In [13]:
from tools import kg_tools
kg_tools._driver.cache_clear()
pd.DataFrame(kg_tools.link_entities("tiểu đường type 2", k=5)["entities"])

id,name,type,score
Disease:dai thao duong tuyp 2,Đái tháo đường tuýp 2,Disease,0.913
Disease:dai thao duong tuyp 1,Đái tháo đường tuýp 1,Disease,0.878
Disease:dai thao duong tuyp 2 nhe,Đái tháo đường tuýp 2 nhẹ,Disease,0.870
RiskFactor:tien su gia dinh co nguoi than truc he mac dai thao duong tuyp 2,Tiền sử gia đình có người thân trực hệ mắc đái tháo đường type 2,RiskFactor,0.711
Disease:tien dai thao duong,Tiền tiểu đường,Disease,0.561


In [14]:
hba1c = kg_tools.explain_feature("HbA1c_level", 6.7)
pd.DataFrame(hba1c["thresholds"])[["condition", "test", "op", "value", "value_max", "unit", "met", "source_url"]]

condition,test,op,value,value_max,unit,met,source_url
HbA1c bình thường,HbA1c,range,4.0,6.0,%,✗,chi-so-hb-a1c-binh-thuong-la-bao-nhieu-khi-nao-nen-lam-xet-nghiem-hb-a1-c
Đái tháo đường,HbA1c,>,6.5,—,%,✓,chi-so-hb-a1c-binh-thuong-la-bao-nhieu-khi-nao-nen-lam-xet-nghiem-hb-a1-c
Tiền đái tháo đường hoặc nguy cơ đái tháo đường,HbA1c,range,5.7,6.4,%,✗,benh-tieu-duong-289
Hạ đường huyết,HbA1c,<,4.0,—,%,✗,chi-so-hb-a1c-binh-thuong-la-bao-nhieu-khi-nao-nen-lam-xet-nghiem-hb-a1-c
Tiền tiểu đường,HbA1c,range,5.7,6.4,%,✗,chi-so-duong-huyet-tien-tieu-duong-la-bao-nhieu
Đái tháo đường tuýp 2,HbA1c,>=,6.5,—,%,✓,tieu-duong-tuyp-2
Đái tháo đường,HbA1c,>=,6.5,—,%,✓,benh-tieu-duong-289
Bình thường,HbA1c,<,5.7,—,%,✗,chi-so-duong-huyet-tien-tieu-duong-la-bao-nhieu
Mức bình thường,HbA1c,<,5.7,—,%,✗,chi-so-hb-a1c-binh-thuong-la-bao-nhieu-khi-nao-nen-lam-xet-nghiem-hb-a1-c


In [15]:
glucose = kg_tools.explain_feature("blood_glucose_level", 150)
print(glucose["note"])
pd.DataFrame(glucose["thresholds"])[["condition", "test", "op", "value", "value_max", "unit", "patient_value_converted", "met"]]

Dataset không ghi rõ đường huyết đo lúc đói hay bất kỳ; khi so với ngưỡng phải nêu giả định.


condition,test,op,value,value_max,unit,patient_value_converted,met
Đái tháo đường,Đường huyết bất kỳ,>=,200.0,—,mg/dL,150.00,✗
Đái tháo đường tuýp 2,Đường huyết bất kỳ,>=,200.0,—,mg/dL,150.00,✗
Đái tháo đường,Glucose,>=,11.1,—,mmol/L,8.33,✗
Bình thường,Glucose,<,7.8,—,mmol/L,8.33,✗
Đái tháo đường,Đường huyết lúc đói (FPG),>=,7.0,—,mmol/L,8.33,✓
Đái tháo đường,Đường huyết ở thời điểm bất kỳ,>=,11.1,—,mmol/L,8.33,✗
Đái tháo đường,Đường huyết ở thời điểm bất kỳ,>=,200.0,—,mg/dL,150.00,✗
Đường huyết bình thường lúc đói,Đường huyết lúc đói,range,70.0,99,mg/dL,150.00,✗
Rối loạn đường huyết lúc đói,Đường huyết lúc đói,range,100.0,126,mg/dL,150.00,✗
Đái tháo đường tuýp 2,Đường huyết lúc đói,>=,126.0,—,mg/dL,150.00,✓


In [16]:
pd.DataFrame(kg_tools.neighbors("Đái tháo đường tuýp 2", ["HAS_COMPLICATION"])["facts"])[["fact", "quote"]]

fact,quote
Đái tháo đường tuýp 2 có thể gây biến chứng Hạ đường huyết,"Các biến chứng cấp tính như: Hôn mê nhiễm toan ceton, hạ glucose máu"
Đái tháo đường tuýp 2 có thể gây biến chứng Bệnh động mạch ngoại biên,"Các biến chứng mạn tính như: Bệnh võng mạc, bệnh thận, bệnh lý thần kinh, đau thắt ngực và nhồi máu cơ tim , thiếu máu não thoáng qua và đột quỵ, bệnh động mạch ngoại biên"
Đái tháo đường tuýp 2 có thể gây biến chứng Đau thắt ngực và nhồi máu cơ tim,"Các biến chứng mạn tính như: Bệnh võng mạc, bệnh thận, bệnh lý thần kinh, đau thắt ngực và nhồi máu cơ tim"
Đái tháo đường tuýp 2 có thể gây biến chứng Suy thận,hoặc suy thận
Đái tháo đường tuýp 2 có thể gây biến chứng Bệnh thận,"Các biến chứng mạn tính như: Bệnh võng mạc, bệnh thận"
Đái tháo đường tuýp 2 có thể gây biến chứng Đột quỵ,đột quỵ
Đái tháo đường tuýp 2 có thể gây biến chứng Hôn mê tăng glucose máu không nhiễm toan ceton,"Các biến chứng cấp tính như: Hôn mê nhiễm toan ceton, hạ glucose máu, hôn mê tăng glucose máu không nhiễm toan ceton"
Đái tháo đường tuýp 2 có thể gây biến chứng Các bệnh nhiễm trùng cấp,"Các biến chứng cấp tính như: Hôn mê nhiễm toan ceton, hạ glucose máu, hôn mê tăng glucose máu không nhiễm toan ceton, hôn mê nhiễm toan Lactic, các bệnh nhiễm trùng cấp,…"
Đái tháo đường tuýp 2 có thể gây biến chứng Đục thuỷ tinh thể,đục thủy tinh thể
Đái tháo đường tuýp 2 có thể gây biến chứng Cao huyết áp,cao huyết áp


In [17]:
pd.DataFrame(kg_tools.drug_info("Metformin")["facts"])[["fact", "quote"]]

fact,quote
Metformin chống chỉ định ở Bệnh phổi thiếu oxygen mạn tính,"Bệnh nhân có bệnh gan/tim mạch nặng, bệnh hô hấp nặng với giảm oxygen huyết; suy tim sung huyết, trụy tim mạch, nhồi máu cơ tim cấp tính ; bệnh phổi thiếu oxygen mạn tính; nhiễm khuẩn nặng, nhiễm khuẩn huyết; mất bù chuyển hóa cấp tính; hoại thư, nghiện rượu, thiếu dinh dưỡng."
Metformin có tác dụng phụ Mày đay,"Chán ăn, buồn nôn, nôn , tiêu chảy, đầy thượng vị, táo bón, ợ nóng; ban, mày đay, cảm thụ với ánh sáng; giảm nồng độ vitamin B 12 ."
Metformin chống chỉ định ở Bệnh hô hấp nặng với giảm oxygen huyết,"Bệnh nhân có bệnh gan/tim mạch nặng, bệnh hô hấp nặng với giảm oxygen huyết; suy tim sung huyết, trụy tim mạch, nhồi máu cơ tim cấp tính ; bệnh phổi thiếu oxygen mạn tính; nhiễm khuẩn nặng, nhiễm khuẩn huyết; mất bù chuyển hóa cấp tính; hoại thư, nghiện rượu, thiếu dinh dưỡng."
Metformin chống chỉ định ở Nhiễm acid chuyển hóa mạn tính,"Bệnh nhân nhiễm acid chuyển hóa cấp tính hoặc mạn tính, có hoặc không có hôn mê (kể cả nhiễm acid – ceton do đái tháo đường)."
Metformin chống chỉ định ở Nhiễm acid chuyển hóa cấp tính,"Bệnh nhân nhiễm acid chuyển hóa cấp tính hoặc mạn tính, có hoặc không có hôn mê (kể cả nhiễm acid – ceton do đái tháo đường)."
Metformin chống chỉ định ở Nhiễm khuẩn huyết,"Bệnh nhân có bệnh gan/tim mạch nặng, bệnh hô hấp nặng với giảm oxygen huyết; suy tim sung huyết, trụy tim mạch, nhồi máu cơ tim cấp tính ; bệnh phổi thiếu oxygen mạn tính; nhiễm khuẩn nặng, nhiễm khuẩn huyết; mất bù chuyển hóa cấp tính; hoại thư, nghiện rượu, thiếu dinh dưỡng."
Metformin chống chỉ định ở Hoại thư,"Bệnh nhân có bệnh gan/tim mạch nặng, bệnh hô hấp nặng với giảm oxygen huyết; suy tim sung huyết, trụy tim mạch, nhồi máu cơ tim cấp tính ; bệnh phổi thiếu oxygen mạn tính; nhiễm khuẩn nặng, nhiễm khuẩn huyết; mất bù chuyển hóa cấp tính; hoại thư, nghiện rượu, thiếu dinh dưỡng."
Metformin chống chỉ định ở Nhiễm acid – ceton do đái tháo đường,"Bệnh nhân nhiễm acid chuyển hóa cấp tính hoặc mạn tính, có hoặc không có hôn mê (kể cả nhiễm acid – ceton do đái tháo đường)."
Metformin chống chỉ định ở Nhiễm trùng nặng,"Bệnh nhân có bệnh gan/tim mạch nặng, bệnh hô hấp nặng với giảm oxygen huyết; suy tim sung huyết, trụy tim mạch, nhồi máu cơ tim cấp tính ; bệnh phổi thiếu oxygen mạn tính; nhiễm khuẩn nặng, nhiễm khuẩn huyết; mất bù chuyển hóa cấp tính; hoại thư, nghiện rượu, thiếu dinh dưỡng."
Metformin chống chỉ định ở Nhồi máu cơ tim cấp tính,"Bệnh nhân có bệnh gan/tim mạch nặng, bệnh hô hấp nặng với giảm oxygen huyết; suy tim sung huyết, trụy tim mạch, nhồi máu cơ tim cấp tính ; bệnh phổi thiếu oxygen mạn tính; nhiễm khuẩn nặng, nhiễm khuẩn huyết; mất bù chuyển hóa cấp tính; hoại thư, nghiện rượu, thiếu dinh dưỡng."


## Bước 8 — Mẫu chấm precision

Lấy ngẫu nhiên 50 quan hệ (seed cố định) ra `processed/precision_sample.csv`. Cột `correct` = 1 nếu nguồn (câu trích và
ngữ cảnh ngay quanh nó) thật sự thể hiện quan hệ đó, đúng chiều, đúng loại, đúng ngưỡng; 0 nếu sai. Cột `ly_do` ghi lý do,
`nguoi_cham` ghi ai chấm: bản hiện tại do Claude chấm nháp, người viết kiểm tra lại (các dòng ghi "(cân nhắc)" là ca ranh
giới) rồi sửa cột `correct` nếu không đồng ý và chạy lại ô dưới. Mục tiêu ≥ 85%.

In [18]:
sample_path = config.PROCESSED_DIR / "precision_sample.csv"
names = {e["id"]: e["name"] for e in graph["entities"]}
pool = [r for r in graph["relations"] if r["relation"] != "MAPS_TO"]
if not sample_path.exists():
    random.seed(42)
    rows = [{"head": names[r["head"]], "relation": r["relation"], "tail": names[r["tail"]],
             "props": json.dumps(r["props"], ensure_ascii=False) if r["props"] else "", "evidence": r["evidence"][0],
             "source_url": r["source_urls"][0], "correct": ""} for r in random.sample(pool, min(50, len(pool)))]
    pd.DataFrame(rows).to_csv(sample_path, index=False)
labels = pd.read_csv(sample_path)
filled = pd.to_numeric(labels["correct"], errors="coerce").dropna()
print(f"Đã chấm {len(filled)}/{len(labels)} quan hệ" + (f" → precision = {filled.mean():.3f}" if len(filled) else ""))

Đã chấm 50/50 quan hệ → precision = 0.940
